# 🧪 CryptoFL — Rodar TODOS os testes no Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luanmantegazine/cryptoFL/blob/claude/todos-script-colab-t0xh5z/colab_todos.ipynb)

Este notebook executa **todos os experimentos** do projeto de novo e, no final, **plota os gráficos com legendas em inglês**:

1. **Scaling** (`scaling_experiment.py`) — FedAvg vs FedProx × número de clientes.
2. **Ablation** (`ablation_experiment.py`) — baseline vs no_ipfs vs full.
3. **Security** (`security_experiment.py`) — robustez a clientes maliciosos.

O código detecta CUDA sozinho (`flower_fl/client.py`). **Antes de tudo:** menu
**Runtime → Change runtime type → Hardware accelerator → GPU (T4)** → *Save*.

Depois rode as células em ordem (**Runtime → Run all**). Os gráficos finais (célula 7)
são regerados a partir dos `*_summary.json` com **rótulos/legendas em inglês**.

## 0) Confirmar a GPU

In [ ]:
import torch, subprocess
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\u26a0\ufe0f  SEM GPU! Runtime > Change runtime type > GPU e rode de novo (roda em CPU, mas lento).')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 1) Clonar o repositório + instalar dependências

In [ ]:
import os
os.chdir('/content')
BRANCH = 'claude/todos-script-colab-t0xh5z'
!test -d cryptoFL || git clone -b $BRANCH https://github.com/Luanmantegazine/cryptoFL.git
%cd /content/cryptoFL
!pip install -q 'flwr==1.7.0' 'numpy<2.0'
!mkdir -p logs results
print('\u2705 ambiente pronto')

## 2) Teste rápido (~1 min) — confirma que o pipeline conecta e treina

In [ ]:
!KMP_DUPLICATE_LIB_OK=TRUE python scaling_experiment.py \
    --clients-list 2 --rounds 1 --aggregators fedavg --repetitions 1 \
    --dataset mnist --model mnistnet --output-dir results/_smoke

## 3) Experimento de SCALING — FedAvg vs FedProx × nº de clientes
Gera `results/scaling_mnist/scaling_summary.json`. Em GPU, ~15–25 min.

> Quer também o CIFAR-10/ResNet-18? Descomente a 2ª linha (mais demorado, ~30–90 min).

In [ ]:
!KMP_DUPLICATE_LIB_OK=TRUE python scaling_experiment.py \
    --clients-list 2,4,6,8,10 --rounds 3 --aggregators fedavg,fedprox --repetitions 3 \
    --dataset mnist --model mnistnet --output-dir results/scaling_mnist 2>&1 | tee logs/scaling_mnist.log

# CIFAR-10 / ResNet-18 (opcional, mais demorado) — descomente para rodar:
# !KMP_DUPLICATE_LIB_OK=TRUE python scaling_experiment.py \
#     --clients-list 2,4,6,8,10 --rounds 3 --aggregators fedavg,fedprox --repetitions 3 \
#     --dataset cifar10 --model resnet18 --alpha 0.5 --output-dir results/scaling_cifar 2>&1 | tee logs/scaling_cifar.log

## 4) Experimento de ABLATION — baseline vs no_ipfs vs full
Gera `results/ablation/ablation_summary.json`.

In [ ]:
!KMP_DUPLICATE_LIB_OK=TRUE python ablation_experiment.py \
    --clients 3 --rounds 3 --modes baseline,no_ipfs,full \
    --output-dir results/ablation 2>&1 | tee logs/ablation.log

## 5) Experimento de SECURITY — robustez a clientes maliciosos
Gera `results/security/security_summary.json`.

In [ ]:
!KMP_DUPLICATE_LIB_OK=TRUE python security_experiment.py \
    --clients 5 --rounds 3 --malicious-pct 0.0,0.25,0.5 --repetitions 1 \
    --attack-type noise --mode full --output-dir results/security 2>&1 | tee logs/security.log

## 6) Tabelas SUMMARY (texto)

In [ ]:
import json
from pathlib import Path

def show_scaling(p):
    s = json.load(open(p)); cfg = s['config']; r = s['results']
    print(f"== SCALING | {cfg['dataset'].upper()} | {cfg['aggregators']} | N={cfg['clients_list']} | reps={cfg['repetitions']} ==")
    h = f"{'agg':<9}{'N':>4}{'mean_acc':>11}{'std_acc':>10}{'mean_time_s':>13}{'n_runs':>8}"
    print(h); print('-'*len(h))
    for n in cfg['clients_list']:
        for a in cfg['aggregators']:
            e = r.get(str(n), {}).get(a)
            if e:
                print(f"{a:<9}{n:>4}{e['mean_accuracy']:>11.4f}{e['std_accuracy']:>10.4f}{e['mean_time_s']:>13.2f}{e['n_runs']:>8}")
    print()

def show_ablation(p):
    s = json.load(open(p)); r = s['results']
    print('== ABLATION ==')
    h = f"{'mode':<10}{'final_acc':>11}{'mean_time_s':>13}{'total_gas_eth':>16}"
    print(h); print('-'*len(h))
    for mode, v in r.items():
        print(f"{mode:<10}{v['final_accuracy']:>11.4f}{v['mean_round_time_s']:>13.2f}{v['total_gas_eth']:>16.8f}")
    print()

def show_security(p):
    s = json.load(open(p)); r = s['results']
    print('== SECURITY ==')
    h = f"{'mal_pct':>8}{'n_mal':>6}{'final_acc':>11}{'acc_drop':>10}{'flagged':>9}"
    print(h); print('-'*len(h))
    for k in sorted(r, key=float):
        v = r[k]
        print(f"{float(k):>8.2f}{v['n_malicious']:>6}{v['final_accuracy']:>11.4f}{v['accuracy_drop']:>10.4f}{v['n_flagged_total']:>9}")
    print()

for p in ['results/scaling_mnist/scaling_summary.json', 'results/scaling_cifar/scaling_summary.json']:
    if Path(p).exists(): show_scaling(p)
if Path('results/ablation/ablation_summary.json').exists(): show_ablation('results/ablation/ablation_summary.json')
if Path('results/security/security_summary.json').exists(): show_security('results/security/security_summary.json')

## 7) 📊 Gráficos finais — legendas em INGLÊS
Regerados a partir dos `*_summary.json`. Salvos em `results/figures_en/` e exibidos inline.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import Image, display

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'axes.titlesize': 13, 'axes.labelsize': 11, 'legend.fontsize': 10, 'figure.dpi': 120})

OUT = Path('results/figures_en'); OUT.mkdir(parents=True, exist_ok=True)
MARKERS = {'fedavg': 'o', 'fedprox': 's'}
saved = []

# ---------- SCALING: accuracy & time vs number of clients ----------
def plot_scaling(path, tag):
    s = json.load(open(path)); cfg = s['config']; res = s['results']
    ns = sorted(int(n) for n in res)
    subt = f"{cfg['dataset'].upper()} · {cfg['model']} · {cfg['rounds']} rounds · {cfg['repetitions']} reps"
    # Accuracy
    plt.figure(figsize=(8, 5))
    for agg in cfg['aggregators']:
        means = [res[str(n)][agg]['mean_accuracy'] for n in ns]
        stds = [res[str(n)][agg]['std_accuracy'] for n in ns]
        plt.errorbar(ns, means, yerr=stds, marker=MARKERS.get(agg, 'o'), capsize=4,
                     linewidth=2, label=agg.upper())
    plt.xlabel('Number of clients'); plt.ylabel('Final accuracy')
    plt.title(f'Final accuracy vs number of clients (FedAvg vs FedProx)\n{subt}')
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    plt.gca().xaxis.set_major_locator(mtick.MaxNLocator(integer=True))
    plt.legend(); plt.tight_layout()
    f1 = OUT / f'{tag}_accuracy_en.png'; plt.savefig(f1, dpi=300); plt.close(); saved.append(f1)
    # Time
    plt.figure(figsize=(8, 5))
    for agg in cfg['aggregators']:
        means = [res[str(n)][agg]['mean_time_s'] for n in ns]
        stds = [res[str(n)][agg].get('std_time_s', 0) for n in ns]
        plt.errorbar(ns, means, yerr=stds, marker=MARKERS.get(agg, 'o'), capsize=4,
                     linewidth=2, label=agg.upper())
    plt.xlabel('Number of clients'); plt.ylabel('Cumulative training time (s)')
    plt.title(f'Training time vs number of clients (FedAvg vs FedProx)\n{subt}')
    plt.gca().xaxis.set_major_locator(mtick.MaxNLocator(integer=True))
    plt.legend(); plt.tight_layout()
    f2 = OUT / f'{tag}_time_en.png'; plt.savefig(f2, dpi=300); plt.close(); saved.append(f2)

for path, tag in [('results/scaling_mnist/scaling_summary.json', 'scaling_mnist'),
                  ('results/scaling_cifar/scaling_summary.json', 'scaling_cifar')]:
    if Path(path).exists():
        plot_scaling(path, tag)

# ---------- ABLATION: bar charts per mode ----------
ap = Path('results/ablation/ablation_summary.json')
if ap.exists():
    res = json.load(open(ap))['results']
    modes = list(res)
    panels = [('final_accuracy', 'Final accuracy', 'Final accuracy per configuration'),
              ('mean_round_time_s', 'Mean round time (s)', 'Mean round time per configuration'),
              ('total_gas_eth', 'Total gas (ETH)', 'Total gas cost per configuration')]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (key, ylabel, title) in zip(axes, panels):
        vals = [res[m][key] for m in modes]
        ax.bar(modes, vals, color=['#1f77b4', '#ff7f0e', '#2ca02c'][:len(modes)])
        ax.set_xlabel('Configuration'); ax.set_ylabel(ylabel); ax.set_title(title)
        for i, v in enumerate(vals):
            ax.text(i, v, f'{v:.3g}', ha='center', va='bottom', fontsize=9)
    fig.suptitle('Ablation study (baseline vs no_ipfs vs full)', fontsize=14)
    fig.tight_layout()
    fa = OUT / 'ablation_en.png'; fig.savefig(fa, dpi=300); plt.close(fig); saved.append(fa)

# ---------- SECURITY: accuracy & accuracy drop vs malicious fraction ----------
sp = Path('results/security/security_summary.json')
if sp.exists():
    res = json.load(open(sp))['results']
    pcts = sorted(res, key=float)
    x = [float(p) * 100 for p in pcts]
    acc = [res[p]['final_accuracy'] for p in pcts]
    acc_std = [res[p].get('std_final_accuracy', 0) for p in pcts]
    drop = [res[p]['accuracy_drop'] for p in pcts]
    drop_std = [res[p].get('std_accuracy_drop', 0) for p in pcts]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].errorbar(x, acc, yerr=acc_std, marker='o', capsize=4, linewidth=2, color='#1f77b4')
    axes[0].set_xlabel('Malicious clients (%)'); axes[0].set_ylabel('Final accuracy')
    axes[0].set_title('Final accuracy vs malicious fraction')
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    axes[1].errorbar(x, drop, yerr=drop_std, marker='s', capsize=4, linewidth=2, color='#d62728')
    axes[1].set_xlabel('Malicious clients (%)'); axes[1].set_ylabel('Accuracy drop')
    axes[1].set_title('Accuracy drop vs malicious fraction')
    fig.suptitle('Security: robustness to malicious clients', fontsize=14)
    fig.tight_layout()
    fs = OUT / 'security_en.png'; fig.savefig(fs, dpi=300); plt.close(fig); saved.append(fs)

print('Saved figures (English legends):')
for f in saved:
    print('  ', f)
    display(Image(str(f)))

## 8) Salvar os resultados (a sessão do Colab é efêmera!)
Baixa um zip com `results/` (inclui `figures_en/`) e `logs/`.

In [ ]:
!zip -qr resultados_todos.zip results logs && echo 'zip pronto'
from google.colab import files
files.download('resultados_todos.zip')

# Opção B — Google Drive (persiste). Descomente para usar:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results logs '/content/drive/MyDrive/cryptoFL_results' && echo 'salvo no Drive'